# 현실의 데이터 37: 채용: 이력서 · 공고 매칭

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/datashop/37_hiring_match.ipynb)

**분야**: 전문 서비스 &nbsp;|&nbsp; **이 분야에서 먼저 보는 문제**: HR 테크 · 텍스트 매칭 · 공정성

> 이력서와 채용 공고 한 쌍. 이 사람은 이 자리에 맞을까?

**데이터**: Resume - Job Description Fit (약 12MB). Hugging Face cnamuangtoun/resume-job-description-fit: 영어 이력서 글 + 채용 공고 글 쌍 약 8천 개, 맞음 (Good Fit) · 가능 (Potential Fit) · 안 맞음 (No Fit) 세 단계.

산업 현장의 문제와 닮은 공개 데이터다. 데이터를 열어 보고, 아래 예시처럼 문제 하나를 정해 기준선과 비교한 뒤, 맨 끝의 **나의 다섯 칸** 을 채운다 (1주 2차시 '문제 정형화와 평가지표').

| 칸 | 이 노트북의 예시 |
|---|---|
| ① 한 행 | (이력서, 공고) 한 쌍 |
| ② 입력 X | 이력서 글 · 공고 글 (단어 빈도, 두 글의 겹침) |
| ③ 출력 y | 맞음 / 가능 / 안 맞음 |
| ④ 평가지표 | 정확도 + 단계별 재현율 |
| ⑤ 기준선 | 가장 흔한 단계 |

**논문 · 출처** (직접 열어 보자: 이 데이터를 누가, 왜, 어떻게 만들었나):
- [Hugging Face 데이터 설명 (resume-job-description-fit)](https://huggingface.co/datasets/cnamuangtoun/resume-job-description-fit)

In [1]:
import io, os, zipfile, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def download(url, path):
    """Download url to path once (skip if it already exists)"""
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    if not os.path.exists(path):
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (ML course data shop)"})
        with urllib.request.urlopen(req, timeout=600) as r, open(path, "wb") as f:
            f.write(r.read())
    return path

## 입장: 데이터 받기

In [2]:
BASE = "https://huggingface.co/api/datasets/cnamuangtoun/resume-job-description-fit/parquet/default/"
train = pd.read_parquet(download(BASE + "train/0.parquet", "data/resume_fit_train.parquet"))
test = pd.read_parquet(download(BASE + "test/0.parquet", "data/resume_fit_test.parquet"))
print(train.shape, test.shape)
print(train["label"].value_counts().to_dict())
print("\nRESUME:", train["resume_text"].iloc[0][:400], "\n\nJOB:", train["job_description_text"].iloc[0][:400])

(6241, 3) (1759, 3)
{'No Fit': 3143, 'Potential Fit': 1556, 'Good Fit': 1542}

RESUME: SummaryHighly motivated Sales Associate with extensive customer service and sales experience. Outgoing sales professional with track record of driving increased sales, improving buying experience and elevating company profile with target market.
Highlights-Soft Skills: Public Speaking, Public Relations, Team Building, Project Management, Procedure writing, Staff Supervision and Management, Ability 

JOB: Net2Source Inc. is an award-winning total workforce solutions company recognized by Staffing Industry Analysts for our accelerated growth of 300% in the last 3 years with over 5500+ employees globally, with over 30+ locations in the US and global operations in 32 countries. We believe in providing staffing solutions to address the current talent gap  Right Talent  Right Time  Right Place  Right Pr


## 열어 보기

글의 길이, 그리고 **두 글에 같이 나오는 단어의 비율** 이 '맞음' 일수록 높을까?

In [3]:
import re
def words(t):
    return set(re.findall(r"[a-z][a-z+#.]{1,}", t.lower()))

def overlap(r, j):
    jw = words(j); return len(words(r) & jw) / max(len(jw), 1)

train["overlap"] = [overlap(r, j) for r, j in zip(train["resume_text"], train["job_description_text"])]
print(train.groupby("label")["overlap"].describe()[["mean", "50%"]].round(3))
print("resume words: median", int(train["resume_text"].str.split().str.len().median()))

                mean    50%
label                      
Good Fit       0.224  0.219
No Fit         0.205  0.192
Potential Fit  0.226  0.215
resume words: median 618


## 문제 하나 골라 보기: (이력서, 공고) → 맞음 / 가능 / 안 맞음

이력서 · 공고를 각각 단어 빈도 (TF-IDF) 로 바꾸고, 두 글의 코사인 유사도를 더해 로지스틱회귀. 데이터가 나눠 둔 학습 · 시험을 그대로 쓴다.

In [4]:
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

vec = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), sublinear_tf=True, stop_words="english")
vec.fit(pd.concat([train["resume_text"], train["job_description_text"]]))

def pair_features(d, vec):
    R, J = vec.transform(d["resume_text"]), vec.transform(d["job_description_text"])
    cos = np.asarray(R.multiply(J).sum(1))
    return hstack([R, J, csr_matrix(cos * 10)]).tocsr()

m = LogisticRegression(C=2, max_iter=3000).fit(pair_features(train, vec), train["label"])
pred = m.predict(pair_features(test, vec))
most = train["label"].value_counts().idxmax()
print(f"baseline (always '{most}'): accuracy {np.mean(test['label'] == most):.1%}")
print(f"TF-IDF pair + logistic : accuracy {np.mean(test['label'] == pred):.1%}")
print("recall per label:", pd.Series(test["label"].values == pred).groupby(test["label"].values).mean().round(2).to_dict())

baseline (always 'No Fit'): accuracy 48.7%
TF-IDF pair + logistic : accuracy 53.4%
recall per label: {'Good Fit': 0.26, 'No Fit': 0.78, 'Potential Fit': 0.35}


**관찰**: 단어가 겹치는지만으로는 '맞음' 을 잘 가려내지 못한다. 같은 일을 다른 말로 쓰기도 하고 (예: ML ↔ machine learning), '가능' 은 사람이 매겨도 애매하다. 문장 임베딩 (14주차) 을 쓰면 나아진다.

## 다른 문제로 바꿔 보기

- **공정성**: 이력서의 이름 · 학교 · 성별을 보고 모델이 판단한다면? 실제로 이 문제로 채용 AI 를 접은 회사가 있다. 무엇을 입력에서 빼야 할까, 빼면 충분할까?
- **검색 문제로**: 공고 하나에 지원자 1,000명. 출력 = 면접 볼 10명의 순서. 지표 = 상위 10명 중 '맞음' 의 수.

### 나의 다섯 칸

아래 표를 직접 채워 보자. 위의 예시와 다르게 정해도 된다.

| 칸 | 내가 정한 것 |
|---|---|
| ① 한 행은 무엇? | |
| ② 입력 X (예측할 때 알 수 있는 것만) | |
| ③ 출력 y | |
| ④ 평가지표 | |
| ⑤ 기준선 | |